In [ ]:
import json
import os

import medmnist
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from bayesian_torch.models.dnn_to_bnn import dnn_to_bnn, get_kl_loss
from medmnist import INFO
from torch.utils.data import DataLoader

LR = 0.001
EPOCHS = 50
BATCH_SIZE = 128
MILESTONES = [20, 35, 45]
GAMMA = 0.5
MC_SAMPLES = 100         
N_BINS = 15            
N_CURVE_POINTS = 50      

DATASETS = ["pathmnist", "dermamnist"]
ARCHS = ["effnet", "resnet18"]

CKPT_DIR = "checkpoints"
RESULTS_PATH = "results.json"

os.makedirs(CKPT_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

BNN_PRIOR = {
    "prior_mu": 0.0,
    "prior_sigma": 1.0,
    "posterior_mu_init": 0.0,
    "posterior_rho_init": -3.0,
    "type": "Reparameterization",
    "moped_enable": True,
    "moped_delta": 0.1,
}

#Data

def get_data(dataset_name):
    info = INFO[dataset_name]
    DataClass = getattr(medmnist, info["python_class"])
    n_channels = info["n_channels"]
    n_classes = len(info["label"])

    transform_train = transforms.Compose([
        transforms.ToTensor(),
        transforms.RandomHorizontalFlip(),
        transforms.Normalize(mean=[0.5] * n_channels, std=[0.5] * n_channels),
    ])
    transform_eval = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5] * n_channels, std=[0.5] * n_channels),
    ])

    trainset = DataClass(split="train", download=True, size=28, transform=transform_eval)
    trainset_eval = DataClass(split="train", download=True, size=28, transform=transform_eval)
    valset = DataClass(split="val", download=True, size=28, transform=transform_eval)
    testset = DataClass(split="test", download=True, size=28, transform=transform_eval)

    trainloader = DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
    trainloader_eval = DataLoader(trainset_eval, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
    valloader = DataLoader(valset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
    testloader = DataLoader(testset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

    return trainset, trainloader, trainloader_eval, valloader, testloader, n_classes


#Models
def build_effnet(n_classes):
    net = torchvision.models.efficientnet_b0(weights=None)
    net.classifier[1] = nn.Linear(net.classifier[1].in_features, n_classes)
    return net


def build_resnet18(n_classes):
    net = torchvision.models.resnet18(weights=None)
    net.fc = nn.Linear(net.fc.in_features, n_classes)
    return net


BUILDERS = {"effnet": build_effnet, "resnet18": build_resnet18}


def make_bayesian(net, moped_enable):
    prior = dict(BNN_PRIOR)
    prior["moped_enable"] = moped_enable
    dnn_to_bnn(net, prior)
    return net

#Checkpoint
def checkpoint_path(key):
    return os.path.join(CKPT_DIR, f"{key}.pth")


def load_checkpoint_if_exists(key, net, optimizer):
    path = checkpoint_path(key)
    if os.path.exists(path):
        ckpt = torch.load(path, map_location=device)
        net.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        return ckpt
    return None


def save_checkpoint(key, net, optimizer, epoch, train_losses, val_losses):
    torch.save({
        "model_state_dict": net.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "epoch": epoch,
        "train_losses": train_losses,
        "val_losses": val_losses,
    }, checkpoint_path(key))


def load_results():
    if os.path.exists(RESULTS_PATH):
        with open(RESULTS_PATH) as f:
            return json.load(f)
    return {}


def save_result(key, entry):
    results = load_results()
    results[key] = entry
    with open(RESULTS_PATH, "w") as f:
        json.dump(results, f, indent=2)


#Training
def train_model(key, net, trainloader, valloader, trainset_len, is_bayesian):
    net.to(device)
    optimizer = optim.Adam(net.parameters(), lr=LR)
    criterion = nn.CrossEntropyLoss()

    ckpt = load_checkpoint_if_exists(key, net, optimizer)
    if ckpt is not None:
        start_epoch = ckpt["epoch"] + 1
        train_losses = ckpt["train_losses"]
        val_losses = ckpt["val_losses"]
        print(f"[{key}] reprise à l'époque {start_epoch}/{EPOCHS}")
    else:
        start_epoch = 0
        train_losses, val_losses = [], []

    scheduler = optim.lr_scheduler.MultiStepLR(
        optimizer, milestones=MILESTONES, gamma=GAMMA, last_epoch=start_epoch - 1
    )

    if start_epoch >= EPOCHS:
        print(f"[{key}] déjà entraîné ({start_epoch}/{EPOCHS} époques), entraînement sauté.")
        return net

    for epoch in range(start_epoch, EPOCHS):
        net.train()
        running_loss_train = 0.0
        for data in trainloader:
            inputs, labels = data[0].to(device), data[1].to(device)
            labels = labels.squeeze(1).long()
            optimizer.zero_grad()
            outputs = net(inputs)
            loss = criterion(outputs, labels)
            if is_bayesian:
                loss = loss + get_kl_loss(net) / trainset_len
            loss.backward()
            optimizer.step()
            running_loss_train += loss.item()
        train_losses.append(running_loss_train / len(trainloader))

        net.eval()
        running_loss_val = 0.0
        with torch.no_grad():
            for data in valloader:
                inputs, labels = data[0].to(device), data[1].to(device)
                labels = labels.squeeze(1).long()
                outputs = net(inputs)
                loss = criterion(outputs, labels)
                if is_bayesian:
                    loss = loss + get_kl_loss(net) / trainset_len
                running_loss_val += loss.item()
        val_losses.append(running_loss_val / len(valloader))

        scheduler.step()

        print(f"[{key}] époque {epoch + 1}/{EPOCHS} - train {train_losses[-1]:.4f} - val {val_losses[-1]:.4f}")

        # Checkpoint après CHAQUE époque -> reprise possible à tout moment.
        save_checkpoint(key, net, optimizer, epoch, train_losses, val_losses)

    return net


#Evaluation
def get_probs_labels_deterministic(loader, net):
    net.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for data in loader:
            inputs, labels = data[0].to(device), data[1].to(device)
            labels = labels.squeeze(1).long()
            outputs = net(inputs)
            probs = F.softmax(outputs, dim=1)
            all_probs.append(probs.cpu().numpy())
            all_labels.append(labels.cpu().numpy())
    return np.concatenate(all_probs), np.concatenate(all_labels)


def get_mc_probs_labels(loader, net, num_mc_samples):
    """num_mc_samples passes forward par batch (les couches Reparameterization
    re-sample à chaque appel, même en eval()). Retourne mc_probs (K, N, C) et labels (N,)."""
    net.eval()
    all_labels = []
    mc_probs_batches = []
    with torch.no_grad():
        for data in loader:
            inputs, labels = data[0].to(device), data[1].to(device)
            labels = labels.squeeze(1).long()
            batch_probs = []
            for _ in range(num_mc_samples):
                outputs = net(inputs)
                probs = F.softmax(outputs, dim=1)
                batch_probs.append(probs.cpu().numpy())
            mc_probs_batches.append(np.stack(batch_probs, axis=0))  # (K, batch, C)
            all_labels.append(labels.cpu().numpy())
    mc_probs = np.concatenate(mc_probs_batches, axis=1)  # (K, N, C)
    labels = np.concatenate(all_labels)
    return mc_probs, labels


def compute_ace(confidences, predicted, labels, n_bins=N_BINS):
    """Adaptive Calibration Error : bins à effectif égal sur la confiance.
    Retourne (ace, confs_bin, accs_bin) où ace = moyenne de |acc_bin - conf_bin|."""
    order = np.argsort(confidences)
    bins = np.array_split(order, n_bins)
    confs_bin, accs_bin = [], []
    for b in bins:
        if len(b) == 0:
            continue
        confs_bin.append(float(confidences[b].mean()))
        accs_bin.append(float((predicted[b] == labels[b]).mean()))
    confs_bin, accs_bin = np.array(confs_bin), np.array(accs_bin)
    ace = float(np.mean(np.abs(accs_bin - confs_bin)))
    return ace, confs_bin.tolist(), accs_bin.tolist()


def compute_auce(uncertainties, predicted, labels, n_bins=N_BINS):
    """Adaptive Uncertainty Calibration Error : bins à effectif égal sur l'incertitude normalisée.
    Retourne (auce, uncs_bin, errs_bin) où auce = moyenne de |err_bin - unc_bin|."""
    order = np.argsort(uncertainties)
    bins = np.array_split(order, n_bins)
    uncs_bin, errs_bin = [], []
    for b in bins:
        if len(b) == 0:
            continue
        uncs_bin.append(float(uncertainties[b].mean()))
        errs_bin.append(float((predicted[b] != labels[b]).mean()))
    uncs_bin, errs_bin = np.array(uncs_bin), np.array(errs_bin)
    auce = float(np.mean(np.abs(errs_bin - uncs_bin)))
    return auce, uncs_bin.tolist(), errs_bin.tolist()


def confidence_vs_accuracy_curve(mean_probs, predicted, labels, n_points=N_CURVE_POINTS):
    thresholds = np.linspace(0, 1, n_points)
    p_accurate_given_confident = []
    for t in thresholds:
        mask = mean_probs.max(axis=-1) >= t
        if mask.sum() > 0:
            p_accurate_given_confident.append(float((predicted[mask] == labels[mask]).mean()))
        else:
            p_accurate_given_confident.append(None)
    return thresholds.tolist(), p_accurate_given_confident


def uncertain_when_inaccurate_curve(predictive_uncertainty, predicted, labels, n_points=N_CURVE_POINTS):
    thresholds_u = np.linspace(0, predictive_uncertainty.max(), n_points)
    inaccurate_mask = predicted != labels
    p_uncertain_given_inaccurate = []
    for t in thresholds_u:
        if inaccurate_mask.sum() > 0:
            p_uncertain_given_inaccurate.append(float((predictive_uncertainty[inaccurate_mask] >= t).mean()))
        else:
            p_uncertain_given_inaccurate.append(None)
    return thresholds_u.tolist(), p_uncertain_given_inaccurate


def evaluate_classic(net, trainloader_eval, valloader, testloader):
    train_probs, train_labels = get_probs_labels_deterministic(trainloader_eval, net)
    val_probs, val_labels = get_probs_labels_deterministic(valloader, net)
    test_probs, test_labels = get_probs_labels_deterministic(testloader, net)

    acc_train = float((train_probs.argmax(axis=1) == train_labels).mean())
    acc_val = float((val_probs.argmax(axis=1) == val_labels).mean())
    acc_test = float((test_probs.argmax(axis=1) == test_labels).mean())

    mean_probs = test_probs
    all_predicted = mean_probs.argmax(axis=1)
    all_labels = test_labels
    num_classes = mean_probs.shape[1]

    predictive_uncertainty = -np.sum(mean_probs * np.log(mean_probs + 1e-12), axis=1)
    normalized_uncertainty = predictive_uncertainty / np.log(num_classes)

    ace, ace_confs, ace_accs = compute_ace(mean_probs.max(axis=-1), all_predicted, all_labels)
    auce, auce_uncs, auce_errs = compute_auce(normalized_uncertainty, all_predicted, all_labels)
    cva_t, cva_v = confidence_vs_accuracy_curve(mean_probs, all_predicted, all_labels)
    uwi_t, uwi_v = uncertain_when_inaccurate_curve(predictive_uncertainty, all_predicted, all_labels)

    return {
        "is_bayesian": False,
        "acc_train": acc_train,
        "acc_val": acc_val,
        "acc_test": acc_test,
        "ace": ace,
        "auce": auce,
        "entropy_mean": float(predictive_uncertainty.mean()),
        "aleatoric_mean": None,
        "mi_mean": None,
        "num_mc_samples": None,
        "ace_curve": {"confidence": ace_confs, "accuracy": ace_accs},
        "auce_curve": {"uncertainty": auce_uncs, "error_rate": auce_errs},
        "confidence_vs_accuracy": {"thresholds": cva_t, "p_accurate_given_confident": cva_v},
        "uncertain_when_inaccurate": {"thresholds": uwi_t, "p_uncertain_given_inaccurate": uwi_v},
    }


def evaluate_bayesian(net, trainloader_eval, valloader, testloader, num_mc_samples):
    train_mc, train_labels = get_mc_probs_labels(trainloader_eval, net, num_mc_samples)
    val_mc, val_labels = get_mc_probs_labels(valloader, net, num_mc_samples)
    test_mc, test_labels = get_mc_probs_labels(testloader, net, num_mc_samples)

    mean_probs_train = train_mc.mean(axis=0)
    mean_probs_val = val_mc.mean(axis=0)
    mean_probs_test = test_mc.mean(axis=0)

    acc_train = float((mean_probs_train.argmax(axis=1) == train_labels).mean())
    acc_val = float((mean_probs_val.argmax(axis=1) == val_labels).mean())
    acc_test = float((mean_probs_test.argmax(axis=1) == test_labels).mean())

    mean_probs = mean_probs_test
    all_predicted = mean_probs.argmax(axis=1)
    all_labels = test_labels
    num_classes = mean_probs.shape[1]

    predictive_uncertainty = -np.sum(mean_probs * np.log(mean_probs + 1e-12), axis=1)
    normalized_uncertainty = predictive_uncertainty / np.log(num_classes)

    sample_entropies = -np.sum(test_mc * np.log(test_mc + 1e-12), axis=2)  # (K, N)
    aleatoric_uncertainty = sample_entropies.mean(axis=0)

    epistemic_uncertainty = predictive_uncertainty - aleatoric_uncertainty

    ace, ace_confs, ace_accs = compute_ace(mean_probs.max(axis=-1), all_predicted, all_labels)
    auce, auce_uncs, auce_errs = compute_auce(normalized_uncertainty, all_predicted, all_labels)
    cva_t, cva_v = confidence_vs_accuracy_curve(mean_probs, all_predicted, all_labels)
    uwi_t, uwi_v = uncertain_when_inaccurate_curve(predictive_uncertainty, all_predicted, all_labels)

    return {
        "is_bayesian": True,
        "acc_train": acc_train,
        "acc_val": acc_val,
        "acc_test": acc_test,
        "ace": ace,
        "auce": auce,
        "entropy_mean": float(predictive_uncertainty.mean()),
        "aleatoric_mean": float(aleatoric_uncertainty.mean()),
        "mi_mean": float(epistemic_uncertainty.mean()),
        "num_mc_samples": num_mc_samples,
        "ace_curve": {"confidence": ace_confs, "accuracy": ace_accs},
        "auce_curve": {"uncertainty": auce_uncs, "error_rate": auce_errs},
        "confidence_vs_accuracy": {"thresholds": cva_t, "p_accurate_given_confident": cva_v},
        "uncertain_when_inaccurate": {"thresholds": uwi_t, "p_uncertain_given_inaccurate": uwi_v},
    }


# Main loop
def run_dataset(dataset_name):
    print(f"\n========== {dataset_name} ==========")
    trainset, trainloader, trainloader_eval, valloader, testloader, n_classes = get_data(dataset_name)
    trainset_len = len(trainset)

    results = load_results()

    for arch in ARCHS:
        key = f"{dataset_name}_{arch}_classic"
        if key in results:
            print(f"[{key}] déjà entraîné ET évalué, on passe entièrement.")
            continue
        net = BUILDERS[arch](n_classes)
        net = train_model(key, net, trainloader, valloader, trainset_len, is_bayesian=False)
        print(f"[{key}] évaluation...")
        entry = evaluate_classic(net, trainloader_eval, valloader, testloader)
        save_result(key, entry)
        results = load_results()

    for arch in ARCHS:
        classic_key = f"{dataset_name}_{arch}_classic"
        for moped in [True, False]:
            tag = "moped" if moped else "nomoped"
            key = f"{dataset_name}_{arch}_{tag}"
            if key in results:
                print(f"[{key}] déjà entraîné ET évalué, on passe entièrement.")
                continue

            net = BUILDERS[arch](n_classes)
            classic_ckpt = torch.load(checkpoint_path(classic_key), map_location=device)
            net.load_state_dict(classic_ckpt["model_state_dict"])
            net = make_bayesian(net, moped_enable=moped)

            net = train_model(key, net, trainloader, valloader, trainset_len, is_bayesian=True)
            print(f"[{key}] évaluation MC ({MC_SAMPLES} samples)...")
            entry = evaluate_bayesian(net, trainloader_eval, valloader, testloader, MC_SAMPLES)
            save_result(key, entry)
            results = load_results()


for dataset_name in DATASETS:
    run_dataset(dataset_name)
print("\nFinished all datasets. Results saved in", RESULTS_PATH)

Device: cuda

========== pathmnist ==========
[pathmnist_effnet_classic] époque 1/50 - train 0.9988 - val 0.8073
[pathmnist_effnet_classic] époque 2/50 - train 0.5777 - val 1.0727
[pathmnist_effnet_classic] époque 3/50 - train 0.4237 - val 0.3623
[pathmnist_effnet_classic] époque 4/50 - train 0.3367 - val 0.4220
[pathmnist_effnet_classic] époque 5/50 - train 0.2789 - val 0.4703
[pathmnist_effnet_classic] époque 6/50 - train 0.2468 - val 0.2797
[pathmnist_effnet_classic] époque 7/50 - train 0.2110 - val 0.5197
[pathmnist_effnet_classic] époque 8/50 - train 0.1991 - val 0.6701
[pathmnist_effnet_classic] époque 9/50 - train 0.1918 - val 0.3127
[pathmnist_effnet_classic] époque 10/50 - train 0.1589 - val 0.4260
[pathmnist_effnet_classic] époque 11/50 - train 0.1534 - val 0.3785
[pathmnist_effnet_classic] époque 12/50 - train 0.1619 - val 0.2055
[pathmnist_effnet_classic] époque 13/50 - train 0.1317 - val 0.9506
[pathmnist_effnet_classic] époque 14/50 - train 0.2732 - val 0.1720
[pathmnist_

100%|██████████| 19.7M/19.7M [00:02<00:00, 7.25MB/s]


[dermamnist_effnet_classic] époque 1/50 - train 1.1437 - val 1.1404
[dermamnist_effnet_classic] époque 2/50 - train 0.9315 - val 0.9045
[dermamnist_effnet_classic] époque 3/50 - train 0.8786 - val 0.8547
[dermamnist_effnet_classic] époque 4/50 - train 0.8316 - val 0.8238
[dermamnist_effnet_classic] époque 5/50 - train 0.7970 - val 0.7939
[dermamnist_effnet_classic] époque 6/50 - train 0.7431 - val 0.8878
[dermamnist_effnet_classic] époque 7/50 - train 0.6996 - val 0.8161
[dermamnist_effnet_classic] époque 8/50 - train 0.6649 - val 0.8115
[dermamnist_effnet_classic] époque 9/50 - train 0.5980 - val 0.8731
[dermamnist_effnet_classic] époque 10/50 - train 0.5929 - val 0.8342
[dermamnist_effnet_classic] époque 11/50 - train 0.5322 - val 0.8923
[dermamnist_effnet_classic] époque 12/50 - train 0.4901 - val 0.8831
[dermamnist_effnet_classic] époque 13/50 - train 0.4666 - val 0.9475
[dermamnist_effnet_classic] époque 14/50 - train 0.4300 - val 0.9463
[dermamnist_effnet_classic] époque 15/50 - 

In [ ]:
"""
Companion script à train_all.py : NE réentraîne rien, recharge juste les checkpoints déjà
entraînés (checkpoints/<key>.pth) et calcule les infos supplémentaires nécessaires pour :
  - accuracy par classe (test)
  - confusion matrix complète (test)
  - entropie / MI moyennes par classe, par split (train/val/test)
  - MI + correctness par échantillon (test, modèles bayésiens seulement) -> pour les courbes
    "MI vs Accuracy" (dataset entier + zoom classes 7&8)

Tout est sauvegardé dans detailed_results.json (une clé par modèle, même convention que
results.json : "<dataset>_<model>", ex. "pathmnist_effnet_moped").

Pré-requis : les 12 checkpoints de train_all.py doivent déjà exister dans checkpoints/.

Usage : python compute_detailed_results.py
"""

import json
import os

import medmnist
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from bayesian_torch.models.dnn_to_bnn import dnn_to_bnn
from medmnist import INFO
from torch.utils.data import DataLoader

# ============================== CONFIG (identique à train_all.py) ==============================

BATCH_SIZE = 128
MC_SAMPLES = 100

DATASETS = ["pathmnist", "dermamnist"]
ARCHS = ["effnet", "resnet18"]

CKPT_DIR = "checkpoints"
DETAILED_RESULTS_PATH = "detailed_results.json"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

BNN_PRIOR = {
    "prior_mu": 0.0,
    "prior_sigma": 1.0,
    "posterior_mu_init": 0.0,
    "posterior_rho_init": -3.0,
    "type": "Reparameterization",
    "moped_enable": True,
    "moped_delta": 0.1,
}


# ============================== DONNÉES ==============================

def get_data(dataset_name):
    info = INFO[dataset_name]
    DataClass = getattr(medmnist, info["python_class"])
    n_channels = info["n_channels"]
    class_names = [info["label"][str(i)] for i in range(len(info["label"]))]

    transform_eval = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5] * n_channels, std=[0.5] * n_channels),
    ])

    trainset_eval = DataClass(split="train", download=True, size=28, transform=transform_eval)
    valset = DataClass(split="val", download=True, size=28, transform=transform_eval)
    testset = DataClass(split="test", download=True, size=28, transform=transform_eval)

    trainloader_eval = DataLoader(trainset_eval, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
    valloader = DataLoader(valset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
    testloader = DataLoader(testset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

    return trainloader_eval, valloader, testloader, class_names


# ============================== MODÈLES (identique à train_all.py) ==============================

def build_effnet(n_classes):
    net = torchvision.models.efficientnet_b0(weights=None)
    net.classifier[1] = nn.Linear(net.classifier[1].in_features, n_classes)
    return net


def build_resnet18(n_classes):
    net = torchvision.models.resnet18(weights=None)
    net.fc = nn.Linear(net.fc.in_features, n_classes)
    return net


BUILDERS = {"effnet": build_effnet, "resnet18": build_resnet18}


def make_bayesian(net, moped_enable):
    prior = dict(BNN_PRIOR)
    prior["moped_enable"] = moped_enable
    dnn_to_bnn(net, prior)
    return net


def checkpoint_path(key):
    return os.path.join(CKPT_DIR, f"{key}.pth")


def load_trained_net(key, net):
    path = checkpoint_path(key)
    if not os.path.exists(path):
        raise FileNotFoundError(f"Checkpoint manquant pour {key} : {path} (lance train_all.py d'abord)")
    ckpt = torch.load(path, map_location=device)
    net.load_state_dict(ckpt["model_state_dict"])
    net.to(device)
    net.eval()
    return net


# ============================== INFÉRENCE ==============================

def get_probs_labels_deterministic(loader, net):
    all_probs, all_labels = [], []
    with torch.no_grad():
        for data in loader:
            inputs, labels = data[0].to(device), data[1].to(device)
            labels = labels.squeeze(1).long()
            outputs = net(inputs)
            probs = F.softmax(outputs, dim=1)
            all_probs.append(probs.cpu().numpy())
            all_labels.append(labels.cpu().numpy())
    return np.concatenate(all_probs), np.concatenate(all_labels)


def get_mc_probs_labels(loader, net, num_mc_samples):
    all_labels = []
    mc_probs_batches = []
    with torch.no_grad():
        for data in loader:
            inputs, labels = data[0].to(device), data[1].to(device)
            labels = labels.squeeze(1).long()
            batch_probs = []
            for _ in range(num_mc_samples):
                outputs = net(inputs)
                probs = F.softmax(outputs, dim=1)
                batch_probs.append(probs.cpu().numpy())
            mc_probs_batches.append(np.stack(batch_probs, axis=0))
            all_labels.append(labels.cpu().numpy())
    mc_probs = np.concatenate(mc_probs_batches, axis=1)
    labels = np.concatenate(all_labels)
    return mc_probs, labels


# ============================== MÉTRIQUES DÉTAILLÉES ==============================

def confusion_matrix(labels, predicted, n_classes):
    cm = np.zeros((n_classes, n_classes), dtype=int)
    np.add.at(cm, (labels, predicted), 1)
    return cm


def per_class_accuracy_from_cm(cm):
    row_sums = cm.sum(axis=1)
    diag = np.diag(cm)
    acc = np.divide(diag, row_sums, out=np.full_like(diag, np.nan, dtype=float), where=row_sums != 0)
    return (acc * 100).tolist()


def entropy_per_sample(probs):
    return -np.sum(probs * np.log(probs + 1e-12), axis=1)


def mean_by_class(values, labels, n_classes):
    out = []
    for c in range(n_classes):
        mask = labels == c
        out.append(float(values[mask].mean()) if mask.sum() > 0 else None)
    return out


def process_classic(dataset_name, arch, class_names, trainloader_eval, valloader, testloader):
    key = f"{dataset_name}_{arch}_classic"
    print(f"[{key}] inference...")
    net = BUILDERS[arch](len(class_names))
    net = load_trained_net(key, net)

    train_probs, train_labels = get_probs_labels_deterministic(trainloader_eval, net)
    val_probs, val_labels = get_probs_labels_deterministic(valloader, net)
    test_probs, test_labels = get_probs_labels_deterministic(testloader, net)

    test_predicted = test_probs.argmax(axis=1)
    cm = confusion_matrix(test_labels, test_predicted, len(class_names))

    entropy_train = entropy_per_sample(train_probs)
    entropy_val = entropy_per_sample(val_probs)
    entropy_test = entropy_per_sample(test_probs)

    return {
        "is_bayesian": False,
        "class_names": class_names,
        "confusion_matrix_test": cm.tolist(),
        "per_class_accuracy_test": per_class_accuracy_from_cm(cm),
        "entropy_per_class": {
            "train": mean_by_class(entropy_train, train_labels, len(class_names)),
            "val": mean_by_class(entropy_val, val_labels, len(class_names)),
            "test": mean_by_class(entropy_test, test_labels, len(class_names)),
        },
        "mi_per_class": {"train": None, "val": None, "test": None},
        "mi_test_persample": None,
        "correct_test_persample": None,
        "labels_test_persample": None,
    }


def process_bayesian(dataset_name, arch, moped, class_names, trainloader_eval, valloader, testloader):
    tag = "moped" if moped else "nomoped"
    key = f"{dataset_name}_{arch}_{tag}"
    print(f"[{key}] inference MC ({MC_SAMPLES} samples)...")
    net = BUILDERS[arch](len(class_names))
    net = make_bayesian(net, moped_enable=moped)
    net = load_trained_net(key, net)

    train_mc, train_labels = get_mc_probs_labels(trainloader_eval, net, MC_SAMPLES)
    val_mc, val_labels = get_mc_probs_labels(valloader, net, MC_SAMPLES)
    test_mc, test_labels = get_mc_probs_labels(testloader, net, MC_SAMPLES)

    def decompose(mc_probs):
        mean_probs = mc_probs.mean(axis=0)
        predictive = entropy_per_sample(mean_probs)                                  # H[E[p]]
        sample_entropies = -np.sum(mc_probs * np.log(mc_probs + 1e-12), axis=2)      # (K, N)
        aleatoric = sample_entropies.mean(axis=0)                                    # E[H[p]]
        epistemic = predictive - aleatoric                                           # MI / BALD
        return mean_probs, predictive, epistemic

    train_mean_probs, train_entropy, train_mi = decompose(train_mc)
    val_mean_probs, val_entropy, val_mi = decompose(val_mc)
    test_mean_probs, test_entropy, test_mi = decompose(test_mc)

    test_predicted = test_mean_probs.argmax(axis=1)
    cm = confusion_matrix(test_labels, test_predicted, len(class_names))
    correct_test = (test_predicted == test_labels).astype(int)

    return {
        "is_bayesian": True,
        "class_names": class_names,
        "confusion_matrix_test": cm.tolist(),
        "per_class_accuracy_test": per_class_accuracy_from_cm(cm),
        "entropy_per_class": {
            "train": mean_by_class(train_entropy, train_labels, len(class_names)),
            "val": mean_by_class(val_entropy, val_labels, len(class_names)),
            "test": mean_by_class(test_entropy, test_labels, len(class_names)),
        },
        "mi_per_class": {
            "train": mean_by_class(train_mi, train_labels, len(class_names)),
            "val": mean_by_class(val_mi, val_labels, len(class_names)),
            "test": mean_by_class(test_mi, test_labels, len(class_names)),
        },
        "mi_test_persample": test_mi.tolist(),
        "correct_test_persample": correct_test.tolist(),
        "labels_test_persample": test_labels.tolist(),
    }


def load_detailed_results():
    if os.path.exists(DETAILED_RESULTS_PATH):
        with open(DETAILED_RESULTS_PATH) as f:
            return json.load(f)
    return {}


def save_detailed_result(key, entry):
    results = load_detailed_results()
    results[key] = entry
    with open(DETAILED_RESULTS_PATH, "w") as f:
        json.dump(results, f, indent=2)


def run_dataset(dataset_name):
    print(f"\n========== {dataset_name} ==========")
    trainloader_eval, valloader, testloader, class_names = get_data(dataset_name)
    results = load_detailed_results()

    for arch in ARCHS:
        key = f"{dataset_name}_{arch}_classic"
        if key in results:
            print(f"[{key}] déjà calculé, on passe.")
            continue
        entry = process_classic(dataset_name, arch, class_names, trainloader_eval, valloader, testloader)
        save_detailed_result(key, entry)
        results = load_detailed_results()

    for arch in ARCHS:
        for moped in [True, False]:
            tag = "moped" if moped else "nomoped"
            key = f"{dataset_name}_{arch}_{tag}"
            if key in results:
                print(f"[{key}] déjà calculé, on passe.")
                continue
            entry = process_bayesian(dataset_name, arch, moped, class_names, trainloader_eval, valloader, testloader)
            save_detailed_result(key, entry)
            results = load_detailed_results()


for dataset_name in DATASETS:
    run_dataset(dataset_name)
print("\nTerminé — tout est dans detailed_results.json")